# Preprocessing QC — Lucas

This notebook lets you visually inspect whether your fMRI preprocessing worked correctly.

**What you need:**
- A preprocessed 4D NIfTI file (e.g. output of fMRIPrep)
- `nilearn`, `nibabel`, `numpy`, `matplotlib`, `scipy` installed

**What this notebook checks:**
1. NIfTI header (shape, voxel size, TR)
2. Schaefer atlas loaded & resampled to your data
3. ROI × Time timeseries (whole brain parcellated)
4. ROI × ROI functional connectivity matrix
5. Time × Time correlation matrix
6. Single ROI: Voxel × Time, Voxel × Voxel FC, Time × Time

---

## 0 — Configuration

**Set your file path here.**

In [ ]:
# ── Edit this ──────────────────────────────────────────────────────────
NIFTI_PATH = '/path/to/your/preprocessed_bold.nii.gz'   # <-- change this

# Schaefer atlas settings
N_ROIS            = 100    # 100, 200, 300, 400 ...
YEO_NETWORKS      = 7      # 7 or 17
ATLAS_RES_MM      = 2      # 1 or 2

# Which ROI index to inspect in detail (1-indexed, 1..N_ROIS)
INSPECT_ROI_IDX   = 1
# ───────────────────────────────────────────────────────────────────────

## 1 — Imports

In [ ]:
import numpy as np
from scipy.stats import zscore
import nibabel as nib
from nilearn import image, datasets
import matplotlib.pyplot as plt
from tqdm import trange

plt.rcParams['figure.dpi'] = 120

## 2 — Load NIfTI & Print Header

In [ ]:
img = nib.load(NIFTI_PATH)

print('=== NIfTI Header ===')
print(img.header)
print()
print(f'Shape      : {img.shape}')
print(f'Voxel size : {img.header.get_zooms()}')
print(f'Data type  : {img.get_data_dtype()}')
print(f'Affine     :')
print(img.affine)

## 3 — Load Schaefer Atlas & Resample to Data Space

In [ ]:
schaefer    = datasets.fetch_atlas_schaefer_2018(
    n_rois=N_ROIS, yeo_networks=YEO_NETWORKS, resolution_mm=ATLAS_RES_MM
)
atlas_img   = image.load_img(schaefer['maps'])
labels      = schaefer['labels']

# Decode bytes if needed
if isinstance(labels[0], bytes):
    labels = [l.decode('utf-8') for l in labels]

print(f'Atlas shape (native): {atlas_img.shape}')
print(f'Number of ROI labels : {len(labels)}')
print(f'First few labels     : {labels[:5]}')

# Resample atlas to match the fMRI data
print('\nResampling atlas to fMRI space...')
atlas_resampled = image.resample_to_img(
    atlas_img, img, interpolation='nearest',
    copy_header=True, force_resample=True
)
print(f'Atlas shape (resampled): {atlas_resampled.shape}')
print(f'fMRI shape             : {img.shape[:3]}')
assert atlas_resampled.shape == img.shape[:3], \
    'Shape mismatch after resampling — check your NIfTI!'
print('Shapes match — atlas resampled successfully.')

## 4 — Extract ROI-Averaged Timeseries

In [ ]:
def get_roi_data(roi_idx, brain_data, atlas):
    """Return voxel × time array for a single ROI (1-indexed)."""
    mask = atlas.get_fdata() == roi_idx
    return np.nan_to_num(brain_data[mask, :])

# Load and z-score the 4D data (z-score along time)
print('Loading and z-scoring 4D data...')
brain_data = zscore(img.get_fdata(), axis=-1)   # shape: X, Y, Z, T
T = brain_data.shape[-1]
print(f'Loaded. Shape: {brain_data.shape}')

# Average within each ROI
roi_ts = np.zeros((N_ROIS, T))   # ROI × Time
for i in trange(1, N_ROIS + 1, desc='Extracting ROI timeseries'):
    vox = get_roi_data(i, brain_data, atlas_resampled)
    if vox.size == 0 or np.all(vox == 0):
        print(f'  Warning: ROI {i} ({labels[i-1]}) is empty after resampling.')
        continue
    roi_ts[i - 1, :] = np.nanmean(vox, axis=0)

# Z-score across time again after averaging
roi_ts = zscore(roi_ts, axis=1)
print(f'ROI timeseries shape: {roi_ts.shape}  (ROIs × Timepoints)')

## 5 — Whole-Brain Parcellated Plots

Three panels:
- **Left** — ROI × Time timeseries (z-scored)
- **Middle** — ROI × ROI functional connectivity matrix
- **Right** — Time × Time correlation matrix

In [ ]:
fc_matrix    = np.corrcoef(roi_ts)          # ROI × ROI
time_time    = np.corrcoef(roi_ts.T)        # T × T

fig, axs = plt.subplots(1, 3, figsize=(22, 5))
fig.suptitle('Whole-Brain Parcellated QC', fontsize=14)

# ROI × Time
im0 = axs[0].imshow(roi_ts, aspect='auto', interpolation='none', cmap='RdBu_r')
axs[0].set_title('ROI × Time timeseries')
axs[0].set_xlabel('Time (TRs)')
axs[0].set_ylabel('ROI')
plt.colorbar(im0, ax=axs[0], label='z-score')

# ROI × ROI FC
im1 = axs[1].imshow(fc_matrix, aspect='equal', interpolation='none',
                    cmap='RdBu_r', vmin=-1, vmax=1)
axs[1].set_title('ROI × ROI FC')
axs[1].set_xlabel('ROI')
axs[1].set_ylabel('ROI')
plt.colorbar(im1, ax=axs[1], label='Pearson r')

# Time × Time
im2 = axs[2].imshow(time_time, aspect='equal', interpolation='none',
                    cmap='RdBu_r', vmin=-1, vmax=1)
axs[2].set_title('Time × Time correlation')
axs[2].set_xlabel('Time (TRs)')
axs[2].set_ylabel('Time (TRs)')
plt.colorbar(im2, ax=axs[2], label='Pearson r')

plt.tight_layout()
plt.show()

## 6 — Single-ROI Inspection (Voxel Level)

Same three panels but now at the voxel level within one ROI.
The ROI to inspect is controlled by `INSPECT_ROI_IDX` at the top.

In [ ]:
roi_label   = labels[INSPECT_ROI_IDX - 1]
vox_data    = get_roi_data(INSPECT_ROI_IDX, brain_data, atlas_resampled)  # voxels × T

print(f'Inspecting ROI {INSPECT_ROI_IDX}: {roi_label}')
print(f'Voxel × Time shape: {vox_data.shape}')

if vox_data.shape[0] == 0:
    print('ERROR: This ROI has no voxels after resampling. Try a different INSPECT_ROI_IDX.')
else:
    # Z-score voxels along time
    vox_data_z = zscore(vox_data, axis=1)

    vox_fc      = np.corrcoef(vox_data_z)       # voxel × voxel
    vox_time    = np.corrcoef(vox_data_z.T)     # T × T

    fig, axs = plt.subplots(1, 3, figsize=(22, 5),
                             gridspec_kw={'width_ratios': [3, 1, 1]})
    fig.suptitle(f'Single ROI: {roi_label}', fontsize=14)

    # Voxel × Time
    im0 = axs[0].imshow(vox_data_z, aspect='auto', interpolation='none', cmap='RdBu_r')
    axs[0].set_title('Voxel × Time timeseries')
    axs[0].set_xlabel('Time (TRs)')
    axs[0].set_ylabel('Voxel')
    plt.colorbar(im0, ax=axs[0], label='z-score')

    # Voxel × Voxel FC
    im1 = axs[1].imshow(vox_fc, aspect='equal', interpolation='none',
                        cmap='RdBu_r', vmin=-1, vmax=1)
    axs[1].set_title('Voxel × Voxel FC')
    axs[1].set_xlabel('Voxel')
    axs[1].set_ylabel('Voxel')
    plt.colorbar(im1, ax=axs[1], label='Pearson r')

    # Time × Time
    im2 = axs[2].imshow(vox_time, aspect='equal', interpolation='none',
                        cmap='RdBu_r', vmin=-1, vmax=1)
    axs[2].set_title('Time × Time correlation')
    axs[2].set_xlabel('Time (TRs)')
    axs[2].set_ylabel('Time (TRs)')
    plt.colorbar(im2, ax=axs[2], label='Pearson r')

    plt.tight_layout()
    plt.show()

---
## What to look for

| Plot | Good sign | Red flag |
|------|-----------|----------|
| ROI × Time | Smooth temporal variation, no hard jumps | Sudden spikes → motion; all-zero rows → empty ROIs |
| ROI × ROI FC | Block structure matching known networks | Uniform or near-zero matrix → bad preprocessing |
| Time × Time | Gradual structure, no repeating stripes | Banded stripes → motion scrubbing artifacts |
| Voxel × Time | Voxels coherent within ROI | Random noise → atlas misalignment |
